In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from activity_qc_plotter import plot_active_area_trajectory, test_genotype_lmm


# ==============================
# 1  DATA LOADING
# ==============================
def load_mea_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df['DIV'] = df['DIV'].astype(int)
    df['Chip_ID'] = df['Chip_ID'].astype(str)
    df['Well'] = df['Well'].astype(str)
    df['NeuronType'] = df['NeuronType'].astype(str)
    df['Active_area'] = pd.to_numeric(df['Active_area'], errors='coerce')
    df['CHIP_WELL'] = df['Chip_ID'] + '_' + df['Well']
    df = df.dropna(subset=['Active_area']).copy()
    return df


# ==============================
# 2  MAD OUTLIERS (per Genotype x DIV)
# ==============================
def detect_outliers_within_groups(
    df: pd.DataFrame,
    value_col: str = 'Active_area',
    min_n: int = 5,
    z_thresh: float = 3.5,
) -> pd.DataFrame:
    df = df.copy()
    df['is_outlier'] = False
    for (nt, div), group in df.groupby(['NeuronType', 'DIV'], sort=False):
        vals = group[value_col].to_numpy(dtype=float)
        if vals.size < min_n:
            continue
        med = np.median(vals)
        mad = np.median(np.abs(vals - med))
        if mad <= 0:
            continue
        modz = 0.6745 * (vals - med) / mad
        df.loc[group.index[np.abs(modz) > z_thresh], 'is_outlier'] = True
    print(f'Marked {int(df["is_outlier"].sum())} MAD-based outliers (|modZ| > {z_thresh})')
    return df


# ==============================
# 3  EXPECTED-CURVE RESIDUAL DIPS (per Genotype)
# ==============================
def _moving_average(y: np.ndarray, window: int = 3) -> np.ndarray:
    if window <= 1:
        return y
    y = np.asarray(y, dtype=float)
    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode='edge')
    return np.convolve(ypad, np.ones(window) / window, mode='valid')


def detect_expected_curve_residuals(
    df: pd.DataFrame,
    value_col: str = 'Active_area',
    late_div_min: int = 14,
    min_div_points: int = 4,
    smooth_window: int = 3,
    resid_modz_thresh: float = 3.5,
    build_expected_from_non_outliers: bool = True,
) -> pd.DataFrame:
    df = df.copy()
    df['expected_y']      = np.nan
    df['resid_expected']  = np.nan
    df['is_bad_residual'] = False
    for nt, sub_all in df.groupby('NeuronType', sort=False):
        sub_fit = (
            sub_all[~sub_all['is_outlier']].copy()
            if build_expected_from_non_outliers and 'is_outlier' in df.columns
            else sub_all.copy()
        )
        div_meds = sub_fit.groupby('DIV')[value_col].median()
        if div_meds.shape[0] < min_div_points:
            continue
        divs        = div_meds.index.to_numpy(dtype=int)
        meds_smooth = _moving_average(div_meds.to_numpy(dtype=float), window=smooth_window)
        expected_map = dict(zip(divs, meds_smooth))
        idx_nt  = df['NeuronType'] == nt
        df.loc[idx_nt, 'expected_y'] = df.loc[idx_nt, 'DIV'].map(expected_map)
        has_exp = idx_nt & df['expected_y'].notna()
        df.loc[has_exp, 'resid_expected'] = (
            df.loc[has_exp, value_col] - df.loc[has_exp, 'expected_y']
        )
        cand  = has_exp & (df['DIV'] >= late_div_min)
        resid = df.loc[cand, 'resid_expected'].to_numpy(dtype=float)
        if resid.size < 5:
            continue
        med_r = np.median(resid)
        mad_r = np.median(np.abs(resid - med_r))
        if mad_r <= 0:
            continue
        modz_r = 0.6745 * (resid - med_r) / mad_r
        df.loc[df.loc[cand].index[modz_r < -resid_modz_thresh], 'is_bad_residual'] = True
    print(f'Flagged {int(df["is_bad_residual"].sum())} residual dips '
          f'(late DIV >= {late_div_min}, modZ < -{resid_modz_thresh})')
    return df


# ==============================
# 4  COMBINE FLAGS
# ==============================
def mark_filtered(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['to_exclude'] = df['is_outlier'] | df['is_bad_residual']
    print(f'Total excluded: {df["to_exclude"].sum()} / {len(df)} '
          f'({df["to_exclude"].mean()*100:.1f}%)')
    return df


# ==============================
# 5  CONFIG  <-- edit paths and group settings here
# ==============================
CSV_PATH  = '/Users/mandarmp/Documents/PHD/HD_MEA_PAPERWITHCDKL5/Fig2/ActivityScan_outputs/Compiled_Activity_Summary.csv'
SAVE_DIR  = '/Users/mandarmp/Documents/PHD/HD_MEA_PAPERWITHCDKL5/Fig2/ActivityScan_outputs/'

GROUP_ORDER = ['MxWT', 'FxHET', 'MxHEMI']
PALETTE     = {'MxWT': '#4C72B0', 'FxHET': '#D55E00', 'MxHEMI': '#A63226'}


# ==============================
# 6  STEP 1 -- load raw data
# ==============================
df_raw = load_mea_data(CSV_PATH)
print(f'Loaded {len(df_raw)} rows | {df_raw["CHIP_WELL"].nunique()} wells | '
      f'DIV {df_raw["DIV"].min()}-{df_raw["DIV"].max()}')


# ==============================
# 7  STEP 2 -- LMM genotype test (run BEFORE any filtering)
#    Model: Active_area ~ Genotype * DIV + (1 | CHIP_WELL)
#    LRT 1: main genotype effect (is one group persistently different?)
#    LRT 2: genotype x DIV interaction (do trajectories diverge?)
# ==============================
fig_lmm, ax_data, ax_fit, lrt = test_genotype_lmm(
    df_raw,
    group_order=GROUP_ORDER,
    palette=PALETTE,
    figsize=(5.5, 5.0),
    save_path=os.path.join(SAVE_DIR, 'ActiveArea_genotype_lmm.svg'),
)
plt.show()


# ==============================
# 8  STEP 3 -- within-genotype QC (MAD + residual dips)
# ==============================
df = detect_outliers_within_groups(df_raw, z_thresh=3.5)
df = detect_expected_curve_residuals(df, late_div_min=14, smooth_window=3,
                                      resid_modz_thresh=3.5)
df = mark_filtered(df)


# ==============================
# 9  STEP 4 -- QC trajectory plot
# ==============================
fig_qc, _ = plot_active_area_trajectory(
    df,
    group_order=GROUP_ORDER,
    palette=PALETTE,
    figsize=(5.5, 3.5),
    show_expected=True,
    show_threshold=50,
    threshold_label='activity',
    ylabel='Active Area (%)',
    save_path=os.path.join(SAVE_DIR, 'ActiveArea_Trajectory_QC.svg'),
)
plt.show()

print('\n=== MAD Outliers ===')
print(df[df['is_outlier']][['NeuronType', 'DIV', 'CHIP_WELL', 'Active_area']]
      .sort_values(['NeuronType', 'DIV']).to_string(index=False))

print('\n=== Residual Dips ===')
print(df[df['is_bad_residual']][['NeuronType', 'DIV', 'CHIP_WELL',
                                   'Active_area', 'expected_y', 'resid_expected']]
      .sort_values(['NeuronType', 'DIV']).to_string(index=False))
